# Local metadata table

Load the shared local structured metadata table produced by local indexing runs.

In [ ]:
import datetime

import numpy
import polars
import xarray

from data_index.analysis.tables import LOCAL_STRUCTURED_METADATA_TABLE
from data_index.analysis.warehouse import ANALYSIS_LOCAL_WAREHOUSE
from data_index.analysis.datasets import (
    DATASET,
    get_dataset_objects_df,
    get_dataset_xarray_dataset,
)

In [ ]:
ANALYSIS_LOCAL_WAREHOUSE

In [ ]:
table = LOCAL_STRUCTURED_METADATA_TABLE.load()
df = table.scan().to_polars()
df

In [ ]:
dataset: DATASET = "station_lucinda_jetty_daily_wetlabs_bb9"

In [ ]:
dataset_df = get_dataset_objects_df(
    df=df,
    dataset=dataset,
    backend="disk",
)

In [ ]:
dataset_df.with_columns(
    polars.col("dimension_sizes").list.get(1).alias("WAVELENGTH_SIZE")
)["WAVELENGTH_SIZE"].value_counts()

In [ ]:
dataset_df["uri"]

In [ ]:
ds = xarray.open_mfdataset(
    dataset_df["uri"],
    combine="nested",
    concat_dim="TIME",
)

In [ ]:
TIME = ds.TIME.values

In [ ]:
TIME = ds.TIME.values
value, count = numpy.unique_counts(TIME)
print("#                TIME:", len(numpy.unique(TIME)))
print("# unique         TIME:", len(TIME))
print("# duplicate == 2 TIME:", (count == 2).sum())
print("# duplicate >  2 TIME:", (count > 2).sum())

In [ ]:
value, count = numpy.unique_counts(TIME)
dupes = []
for value, count in zip(value, count):
    if count > 1:
        dupes.append(value)

times = polars.Series([
    datetime.datetime.fromisoformat(str(v)).time().strftime("%H-%M-%S")
    for v in dupes
])

with polars.Config(tbl_rows=21):
    print(times.value_counts(sort=True))


In [ ]:
value, count = numpy.unique_counts(TIME)
for value, count in zip(value, count):
    if count > 1:
        print(value, count)